In [1]:
import os
import shutil
import tensorflow as tf
import keras
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from keras import layers
from keras.applications import EfficientNetB2
from classification_models.keras import Classifiers
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm

In [2]:
ResNet34, preprocess_input = Classifiers.get("resnet34")

In [3]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 3
NUM_CLASSES = 2
CLASS_NAMES = ["bird", "drone"]

In [5]:
def organizar_dataset(split):
    image_folder = os.path.join(split, 'images')
    images = [f for f in os.listdir(image_folder) if f.lower().endswith(('.jpg', '.jpeg'))]

    for img_name in tqdm(images):
        if img_name.startswith('B'):
            class_name = 'bird'
        elif img_name.startswith('D'):
            class_name = 'drone'
        else:
            continue

        src_img = os.path.join(image_folder, img_name)
        dst_img = os.path.join("data", class_name, img_name)
        shutil.copy(src_img, dst_img)

organizar_dataset('train')
organizar_dataset('valid')

100%|█████████████████████████████████████████████████████████████████████████████| 1740/1740 [00:11<00:00, 156.92it/s]


In [4]:
def load_data(dir_path):
    train_ds = keras.utils.image_dataset_from_directory(
        dir_path,
        validation_split=0.2,
        subset="training",
        seed = 123,
        batch_size = BATCH_SIZE,
        image_size = IMG_SIZE,
        label_mode = "categorical"
    )
    val_ds = keras.utils.image_dataset_from_directory(
        dir_path,
        validation_split=0.2,
        subset="validation",
        seed = 123,
        batch_size = BATCH_SIZE,
        image_size = IMG_SIZE,
        label_mode = "categorical"
    )
    return train_ds, val_ds

train_ds, val_ds = load_data("data")

Found 20063 files belonging to 2 classes.
Using 16051 files for training.
Found 20063 files belonging to 2 classes.
Using 4012 files for validation.


In [5]:
def process_dataset(dataset):
    AUTOTUNE = tf.data.AUTOTUNE
    norm = layers.Rescaling(1./255)
    norm_ds = dataset.map(lambda x, y: (norm(x), y))
    norm_ds = norm_ds.cache().prefetch(buffer_size=AUTOTUNE)
    return norm_ds

train_ds = process_dataset(train_ds)
val_ds = process_dataset(val_ds)

In [6]:
inputs = keras.Input(shape = (*IMG_SIZE, 3))

x = layers.Conv2D(32, (3, 3), activation = "relu", padding = "same")(inputs)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(32, (3, 3), activation = "relu", padding = "same")(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(64, (3, 3), activation = 'relu', padding = 'same')(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(64, (3, 3), activation = 'relu', padding = 'same')(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(128, (3, 3), activation = 'relu', padding = 'same')(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(128, (3, 3), activation = 'relu', padding = 'same')(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Flatten()(x)
x = layers.Dense(256, activation = 'relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation = 'relu')(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(NUM_CLASSES, activation = "sigmoid")(x)


In [21]:
model =  keras.Model(inputs, outputs)
model.compile(
    optimizer = "adam",
    loss = "categorical_crossentropy",
    metrics = ["accuracy", "recall", "f1_score"]
)

history =  model.fit(
    train_ds,
    validation_data = val_ds,
    epochs = EPOCHS,
    batch_size = BATCH_SIZE)

Epoch 1/3
251/251 ━━━━━━━━━━━━━━━━━━━━ 523s 2s/step - accuracy: 0.9570 - f1_score: 0.9579 - loss: 0.1400 - recall: 0.9493 - val_accuracy: 0.8579 - val_f1_score: 0.8573 - val_loss: 0.4375 - val_recall: 0.8497
Epoch 2/3
251/251 ━━━━━━━━━━━━━━━━━━━━ 518s 2s/step - accuracy: 0.9771 - f1_score: 0.9779 - loss: 0.0752 - recall: 0.9744 - val_accuracy: 0.9457 - val_f1_score: 0.9443 - val_loss: 0.4385 - val_recall: 0.9469
Epoch 3/3
251/251 ━━━━━━━━━━━━━━━━━━━━ 520s 2s/step - accuracy: 0.9671 - f1_score: 0.9743 - loss: 0.0532 - recall: 0.9753 - val_accuracy: 0.9422 - val_f1_score: 0.9408 - val_loss: 0.2735 - val_recall: 0.9414


In [7]:
def evaluate_model(model, val_ds):
    y_true = np.concatenate([y for x, y in val_ds], axis = 0)
    y_pred = model.predict(val_ds)
    y_true_classes = np.argmax(y_true, axis = 1)
    y_pred_classes = np.argmax(y_pred, axis = 1)

    accuracy = accuracy_score(y_true_classes, y_pred_classes)
    recall = recall_score(y_true_classes, y_pred_classes, average = 'weighted')
    f1 = f1_score(y_true_classes, y_pred_classes, average = 'weighted')
    conf_matrix = confusion_matrix(y_true_classes, y_pred_classes)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print("\nConfusion Matrix:")
    print(conf_matrix)

In [13]:
evaluate_model(en_model, val_ds)

126/126 ━━━━━━━━━━━━━━━━━━━━ 49s 374ms/step
Accuracy: 0.7283
Recall: 0.7283
F1-score: 0.7310

Confusion Matrix:
[[1315  270]
 [ 820 1607]]


In [8]:
def evaluate(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.numpy().transpose(0, 3, 1, 2)
            images = torch.FloatTensor(images).to(device)
            labels = torch.FloatTensor(labels.numpy()).to(device)
            labels = torch.argmax(labels, dim=1)
            
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Метрики
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    mtrx = confusion_matrix(all_labels, all_preds)
    
    print(f'Accuracy: {accuracy:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1-score: {f1:.4f}')
    print(f'Confusion matrix: {mtrx}')

In [14]:
evaluate(an_model, val_ds)

Accuracy: 0.9798
Recall: 0.9798
F1-score: 0.9797
Confusion matrix: [[1517   68]
 [  13 2414]]


In [9]:
rn_base_model = ResNet34(
    weights = "imagenet",
    include_top = False,
    input_shape = (*IMG_SIZE, 3)
)

rn_base_model.trainable = False
for layer in rn_base_model.layers[-4:]:
    layer.trainable = True 

rn_model = keras.Sequential([
    rn_base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(128, activation = 'relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation = 'sigmoid')
])
rn_model.compile(
    optimizer = "adam",
    loss = "categorical_crossentropy",
    metrics = ["accuracy", "recall", "f1_score"]
)

rn_history = rn_model.fit(
    train_ds,
    validation_data = val_ds,
    epochs = EPOCHS,
    batch_size = BATCH_SIZE
)

Epoch 1/3
502/502 ━━━━━━━━━━━━━━━━━━━━ 221s 432ms/step - accuracy: 0.6925 - f1_score: 0.6733 - loss: 0.5961 - recall: 0.8993 - val_accuracy: 0.7106 - val_f1_score: 0.7106 - val_loss: 0.5507 - val_recall: 0.9808
Epoch 2/3
502/502 ━━━━━━━━━━━━━━━━━━━━ 215s 429ms/step - accuracy: 0.7418 - f1_score: 0.7278 - loss: 0.5186 - recall: 0.9238 - val_accuracy: 0.7789 - val_f1_score: 0.7474 - val_loss: 0.4866 - val_recall: 0.9980
Epoch 3/3
502/502 ━━━━━━━━━━━━━━━━━━━━ 191s 380ms/step - accuracy: 0.7592 - f1_score: 0.7458 - loss: 0.4956 - recall: 0.9365 - val_accuracy: 0.7894 - val_f1_score: 0.7808 - val_loss: 0.4640 - val_recall: 0.9983


In [10]:
an_model = models.alexnet(pretrained=True)

for param in an_model.parameters():
    param.requires_grad = False

for param in an_model.classifier[-4:].parameters():
    param.requires_grad = True

an_model.classifier[6] = nn.Linear(4096, NUM_CLASSES)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
an_model = an_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, an_model.parameters()), lr=0.001)

for epoch in tqdm(range(EPOCHS)):
    an_model.train()
    running_loss = 0.0
    
    for images, labels in train_ds:
        images = images.numpy().transpose(0, 3, 1, 2)
        images = torch.FloatTensor(images).to(device)
        labels = torch.FloatTensor(labels.numpy()).to(device)
        labels = torch.argmax(labels, dim=1)
        
        outputs = an_model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

c:\Users\casch\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\casch\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 3/3 [05:44<00:00, 114.73s/it]


In [11]:
en_base_model = EfficientNetB2(
    weights = "imagenet",
    include_top = False,
    input_shape = (*IMG_SIZE, 3)
)

en_base_model.trainable = False
for layer in rn_base_model.layers[-4:]:
    layer.trainable = True

x = en_base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation = 'relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(NUM_CLASSES, activation = 'sigmoid')(x)

en_model = keras.Model(
    inputs = en_base_model.input,
    outputs = output
)

en_model.compile(
    optimizer = "adam",
    loss = "categorical_crossentropy",
    metrics = ["accuracy", "recall", "f1_score"]
)

en_history = en_model.fit(
    train_ds,
    validation_data = val_ds,
    epochs = EPOCHS,
    batch_size = BATCH_SIZE
)

Epoch 1/3
502/502 ━━━━━━━━━━━━━━━━━━━━ 229s 433ms/step - accuracy: 0.5959 - f1_score: 0.5729 - loss: 0.8707 - recall: 0.6867 - val_accuracy: 0.7111 - val_f1_score: 0.6991 - val_loss: 0.6534 - val_recall: 0.3868
Epoch 2/3
502/502 ━━━━━━━━━━━━━━━━━━━━ 217s 432ms/step - accuracy: 0.6239 - f1_score: 0.5961 - loss: 0.6434 - recall: 0.6182 - val_accuracy: 0.7129 - val_f1_score: 0.7121 - val_loss: 0.6041 - val_recall: 0.6879
Epoch 3/3
502/502 ━━━━━━━━━━━━━━━━━━━━ 215s 429ms/step - accuracy: 0.6340 - f1_score: 0.6055 - loss: 0.6328 - recall: 0.6162 - val_accuracy: 0.7283 - val_f1_score: 0.7269 - val_loss: 0.5995 - val_recall: 0.8938
